In [1]:
import os
import torch
import pandas as pd
import numpy as np
import pickle
from eval import season_performance_with_unlimited_transfers
import json
from model import FPLSequenceModel

In [2]:
base_path = os.getcwd()
base_path

'/Users/bragehs/Documents/FPL_forecast/predictor'

In [3]:
data_path = os.path.join(base_path, 'processed_data')
data_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [4]:
X_test_numeric = torch.load(data_path + '/X_test.pt', weights_only=True)
X_test_static = torch.load(data_path + '/X_static_test.pt', weights_only=True)
y_test = torch.load(data_path + '/y_test.pt', weights_only=True)
test_mapping = pd.read_csv(data_path + '/test_mapping.csv')
player_ids_test = torch.load(data_path + '/test_player_ids.pt', weights_only=True)
pos_ids_test = torch.load(data_path + '/pos_test.pt', weights_only=True)
fixdiff_ids_test = torch.load(data_path + '/fixdiff_test.pt', weights_only=True)

In [5]:
best_model_data = torch.load("best_model_mae.pth", map_location=torch.device('cpu'))

/var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/ipykernel_35937/2216842584.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  best_model_data = torch.load("best_model_mae.

In [6]:
print(best_model_data.keys())
for k, v in best_model_data['model_state_dict'].items():
    if 'embedding' in k:
        print(k, v.shape)

dict_keys(['model_state_dict', 'best_performance', 'hidden_dim', 'lstm_layers', 'dropout'])
position_embedding.weight torch.Size([6, 16])
fixdiff_embedding.weight torch.Size([6, 16])


In [7]:
with open(f"{data_path}/vocab/player_name_to_idx.json") as f:
    name_to_idx = json.load(f)
with open(f"{data_path}/vocab/unk_id.txt") as f:
    unk_id = int(f.read())

In [8]:
print(best_model_data['hidden_dim'])
print(best_model_data['lstm_layers'])

192
2


In [9]:
model = FPLSequenceModel(
            numeric_seq_dim=X_test_numeric.shape[-1],
            static_dim=X_test_static.shape[-1],
            hidden_dim=best_model_data['hidden_dim'],
            lstm_layers=best_model_data["lstm_layers"],
            dropout=0.0,
            position_vocab_size=6,
            position_embed_dim=16,
            fixture_diff_vocab_size=6,
            fixture_diff_embed_dim=16,
        )
model.load_state_dict(best_model_data['model_state_dict'])

<All keys matched successfully>

In [10]:
#print number of parameters in the model
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of parameters in the model: {num_params}")

Number of parameters in the model: 608578


In [11]:
model.eval()

FPLSequenceModel(
  (position_embedding): Embedding(6, 16, padding_idx=0)
  (fixdiff_embedding): Embedding(6, 16, padding_idx=0)
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (locked_dropout_in): LockedDropout()
  (lstm): LSTM(48, 192, num_layers=2, batch_first=True)
  (locked_dropout_out): LockedDropout()
  (attn_pool): AttentionPool(
    (proj): Sequential(
      (0): Linear(in_features=192, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=1, bias=True)
    )
  )
  (out_dropout): Dropout(p=0.0, inplace=False)
  (head_points): Sequential(
    (0): Linear(in_features=590, out_features=192, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=192, out_features=1, bias=True)
  )
)

In [12]:
test = pd.read_csv(data_path + '/test_data.csv')

In [13]:
test.columns

Index(['GW', 'last_1_assists', 'last_3_assists', 'last_5_assists',
       'last_1_bonus', 'last_3_bonus', 'last_5_bonus', 'last_1_creativity',
       'last_3_creativity', 'last_5_creativity', 'last_1_clean_sheets',
       'last_3_clean_sheets', 'last_5_clean_sheets', 'last_1_goals_conceded',
       'last_3_goals_conceded', 'last_5_goals_conceded', 'last_1_goals_scored',
       'last_3_goals_scored', 'last_5_goals_scored', 'last_1_ict_index',
       'last_3_ict_index', 'last_5_ict_index', 'last_1_influence',
       'last_3_influence', 'last_5_influence', 'last_1_minutes',
       'last_3_minutes', 'last_5_minutes', 'last_1_threat', 'last_3_threat',
       'last_5_threat', 'last_1_red_cards', 'last_3_red_cards',
       'last_5_red_cards', 'last_1_yellow_cards', 'last_3_yellow_cards',
       'last_5_yellow_cards', 'last_1_team_score', 'last_3_team_score',
       'last_5_team_score', 'last_1_opponent_score', 'last_3_opponent_score',
       'last_5_opponent_score', 'last_all_assists', 'last_

In [15]:
def forward_model(x_numeric, x_static, pos=None, fixd=None):
    kwargs = {}
    if getattr(model, 'use_position', False) and pos is not None:
        kwargs['pos_ids'] = pos
    if getattr(model, 'use_fixdiff', False) and fixd is not None:
        kwargs['fixdiff_ids'] = fixd
    try:
        return model(x_numeric, x_static, **kwargs) if kwargs else model(x_numeric, x_static)
    except TypeError:
        return model(x_numeric, x_static)

In [16]:
predictions = forward_model(X_test_numeric, X_test_static , pos=pos_ids_test, fixd=fixdiff_ids_test).detach().numpy()
print(predictions.shape)
print(y_test.shape)

(27283, 1)
torch.Size([27283, 1])


In [17]:
def make_predicted_table(y_test, y_pred):
    '''
    Create a DataFrame for LSTM model predictions.
    This needs to keep track of the Gameweek (GW) and player names.
    '''
    test_mapping = pd.read_csv(data_path + '/test_mapping.csv')
    predictions_df = test_mapping.copy()
    predictions_df['actual'] = y_test
    predictions_df['predicted'] = y_pred
    predictions_df['predicted'] = predictions_df['predicted'].where(predictions_df['minutes'] > 0, 0)
    predictions_df.rename(columns={'prediction_gw': 'GW'}, inplace=True)


    # Update the predictions_df reference
    predictions_df = predictions_df.drop_duplicates(subset=['name', 'GW'], keep='last')
    
    return predictions_df


In [18]:
df = make_predicted_table(y_test, predictions)

In [19]:
salah = df[df['name'] == 'mohamed_salah']
salah

,sequence_idx,element,season_x,name,GW,team_x,value,minutes,last_1_goals_scored,last_1_assists,padding_used,position_encoded,actual,predicted
12426,12426,328.0,2024-25,mohamed_salah,1.0,NaN,125.0,90.0,0.00,0.00,4,3.0,14.0,0.656353
12427,12427,328.0,2024-25,mohamed_salah,2.0,NaN,125.0,82.0,0.25,0.25,3,3.0,10.0,2.656297
12428,12428,328.0,2024-25,mohamed_salah,3.0,NaN,126.0,90.0,0.25,0.00,2,3.0,17.0,2.721791
12429,12429,328.0,2024-25,mohamed_salah,4.0,NaN,127.0,90.0,0.25,0.50,1,3.0,2.0,3.021735
12430,12430,328.0,2024-25,mohamed_salah,5.0,NaN,127.0,90.0,0.00,0.00,0,3.0,6.0,2.505668
12431,12431,328.0,2024-25,mohamed_salah,6.0,NaN,128.0,90.0,0.00,0.25,0,3.0,10.0,3.051100
12432,12432,328.0,2024-25,mohamed_salah,7.0,NaN,127.0,72.0,0.25,0.00,0,3.0,3.0,3.087439
12433,12433,328.0,2024-25,mohamed_salah,8.0,NaN,126.0,90.0,0.00,0.00,0,3.0,12.0,2.711574
12434,12434,328.0,2024-25,mohamed_salah,9.0,NaN,126.0,90.0,0.25,0.25,0,3.0,10.0,2.778683
12435,12435,328.0,2024-25,mohamed_salah,10.0,NaN,127.0,90.0,0.25,0.00,0,3.0,9.0,3.062077


In [20]:
print(torch.mean(y_test))
print(torch.var(y_test))    

tensor(1.1469)
tensor(5.3394)


In [21]:
print(np.mean(predictions))
print(np.var(predictions))
print(np.max(predictions))

0.67488027
0.6571421
4.7509484


In [22]:
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

rmse = root_mean_squared_error(y_test, predictions)
print(f"RMSE: {rmse}")

mae = mean_absolute_error(y_test, predictions)
print(f"MAE: {mae}")


RMSE: 2.042497158050537
MAE: 0.8875532746315002


In [23]:
X_test_numeric.shape

torch.Size([27283, 5, 16])

In [25]:
scores, total_score = season_performance_with_unlimited_transfers(
    y_test=y_test,
    predictions=predictions,
)

Players with NaN total_points_last_season: []
Number of NaN values remaining: 0
1.0 :  lukasz_fabianski
2.0 :  sepp_van_den_berg
3.0 :  tyler_dibling
4.0 :  daniel_jebbison
Bench players: ['lukasz_fabianski', 'sepp_van_den_berg', 'tyler_dibling', 'daniel_jebbison']
Bench cost: 170.0
Simulating season with unlimited transfers for 38 gameweeks
Available budget per gameweek: 830.0

--- Gameweek 1.0 ---
Players available for GW 1.0: 668
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/pulp/apis/../solverdir/cbc/osx/i64/cbc /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/782069c8ac8749b8abba0b3449bbedb1-pulp.mps -max -timeMode elapsed -branch -printingOptions all -solution /var/folders/yv/683h12gj4mdc5txzm0lh6j0h0000gn/T/782069c8ac8749b8abba0b3449bbedb1-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 30 COLUMNS
At line 3687 RHS
At line 3713 BOUNDS
At line 

In [26]:
total_score.item()

2245.0

In [27]:
scores

,team,gw_score
0,"[alexander_isak, dejan_kulusevski, fabian_scha...",33.0
1,"[bernardo_veiga_de_carvalho_e_silva, bukayo_sa...",63.0
2,"[bukayo_saka, cristian_romero, david_raya_mart...",85.0
3,"[alisson_ramses_becker, bukayo_saka, erling_ha...",49.0
4,"[amad_diallo, axel_tuanzebe, bukayo_saka, dwig...",46.0
5,"[andre_onana, bryan_mbeumo, danny_welbeck, dio...",36.0
6,"[cole_palmer, cristian_romero, dejan_kulusevsk...",40.0
7,"[brennan_johnson, bryan_mbeumo, chris_wood, co...",47.0
8,"[chris_wood, cole_palmer, dwight_mcneil, erlin...",68.0
9,"[alex_iwobi, bukayo_saka, chris_wood, cole_pal...",55.0
